# Week 1, Day 2: Dates

Exercises: `day-02.md`

In [2]:
import sys
sys.path.append("../..")

import numpy as np
import pandas as pd
import plotly.express as px

from db import connect

con = connect()

In [14]:
df1 = con.sql("""
select * from orders
""").df()

display(df1.info())
display(df1['order_status'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


None

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 2.1

In [23]:
con.sql("""
select
count(distinct order_id) as orders_count
,date_trunc('month', order_purchase_timestamp) as month
from orders
group by month
order by month
""").df()


,orders_count,month
0,4,2016-09-01
1,324,2016-10-01
2,1,2016-12-01
3,800,2017-01-01
4,1780,2017-02-01
5,2682,2017-03-01
6,2404,2017-04-01
7,3700,2017-05-01
8,3245,2017-06-01
9,4026,2017-07-01


## 2.2

In [27]:
con.sql("""
select 
count(distinct order_id) as orders_count
,date_trunc('month',order_purchase_timestamp) as month 
from orders
group by month
having orders_count < 100
""").df()

,orders_count,month
0,4,2018-10-01
1,16,2018-09-01
2,1,2016-12-01
3,4,2016-09-01


## 2.3

In [42]:
con.sql("""
select
extract('weekday' from order_purchase_timestamp) as week_num
,dayname(order_purchase_timestamp) as weekday
,count(distinct order_id) as orders_count
from orders
group by all
order by week_num asc
""").df()

,week_num,weekday,orders_count
0,0,Sunday,11960
1,1,Monday,16196
2,2,Tuesday,15963
3,3,Wednesday,15552
4,4,Thursday,14761
5,5,Friday,14122
6,6,Saturday,10887


## 2.4

In [45]:
con.sql("""
select
extract('hour' from order_purchase_timestamp) as hour
,count(distinct order_id) as orders_count
from orders
group by all
order by orders_count desc limit 3
""").df()

,hour,orders_count
0,16,6675
1,11,6578
2,14,6569


## 2.5

In [66]:
df1 = con.sql("""
select * from orders
""").df()

df1['month'] = df1['order_purchase_timestamp'].dt.to_period('M')

df1.groupby(['month']).size()

month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64

## 2.6

In [87]:
df2 = con.sql("""
select * from orders
""").df()

df2['order_purchase_timestamp'] = pd.to_datetime(df2['order_purchase_timestamp'])

df2['month_start'] = df2['order_purchase_timestamp'].dt.to_period('M')

df2_agg = (
    df2.groupby(['month_start'], as_index=False)
    .agg(
        orders_count=('order_id','count')
    )
    .sort_values('month_start', ascending=True)
)

df2_agg['order_count_yoy'] = round(df2_agg['orders_count'].pct_change()*100,1)

df2_agg

,month_start,orders_count,order_count_yoy
0,2016-09,4,NaN
1,2016-10,324,8000.0
2,2016-12,1,-99.7
3,2017-01,800,79900.0
4,2017-02,1780,122.5
5,2017-03,2682,50.7
6,2017-04,2404,-10.4
7,2017-05,3700,53.9
8,2017-06,3245,-12.3
9,2017-07,4026,24.1
